In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# TODO: Clone your GitHub repository
# you will require your GitHub Token
from getpass import getpass
token = getpass('Enter your GitHub token:')
!git clone https://{token}@github.com/Saurav51/cs568-project.git


Enter your GitHub token:··········
fatal: destination path 'cs568-project' already exists and is not an empty directory.


**Gemini API setup**

In [ ]:
# Install Google Generative AI SDK
import subprocess

def run(cmd, label=""):
    if label:
        print(f"{label}...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output:
        print(output[-800:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{output[-1200:]}")
    return result

run("pip install -q google-generativeai", "Installing dependencies")

import google.generativeai as genai
from getpass import getpass

MODEL_NAME = "gemini-2.5-flash-lite"

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")
genai.configure(api_key=GEMINI_API_KEY)

print(f"Gemini client ready. Model: {MODEL_NAME}")

**Loading data from files**

In [ ]:
"""Loading data from files"""

# Load personas.py and define project paths
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/cs568-project")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PERSONAS_FILE = DATA_DIR / "personas.py"
if not PERSONAS_FILE.exists():
    raise FileNotFoundError(f"Could not find {PERSONAS_FILE}. Clone the repo first.")

sys.path.insert(0, str(DATA_DIR))
from personas import EXPERT_PERSONAS, NOVICE_PERSONAS, ALL_PERSONAS

print("Loaded personas.py successfully")
print(f"Experts: {len(EXPERT_PERSONAS)}")
print(f"Novices: {len(NOVICE_PERSONAS)}")
print(f"Total:   {len(ALL_PERSONAS)}")
print("First persona:", ALL_PERSONAS[0]["id"], "-", ALL_PERSONAS[0]["name"])

# Load attributes.json from the repo
import json

ATTRIBUTES_FILE = DATA_DIR / "attributes.json"
if not ATTRIBUTES_FILE.exists():
    raise FileNotFoundError(f"Could not find attributes.json at: {ATTRIBUTES_FILE}")

with open(ATTRIBUTES_FILE, "r", encoding="utf-8") as f:
    attributes_data = json.load(f)

ATTRIBUTE_POOL = attributes_data["attribute_pool"]
ATTRIBUTE_DESCRIPTIONS = attributes_data["attribute_descriptions"]
ATTRIBUTE_DIRECTION = attributes_data["attribute_direction"]

print("Loaded attributes.json")
print(f"Number of attributes: {len(ATTRIBUTE_POOL)}")
print("Attributes:", ATTRIBUTE_POOL)

Loaded personas.py successfully
Experts: 7
Novices: 7
Total:   14
First persona: EXP_01 - Marcus Chen
Loaded attributes.json
Number of attributes: 11
Attributes: ['price_usd', 'image_quality_score', 'battery_life_hours', 'optical_zoom_x', 'weight_grams', 'ease_of_use_score', 'has_manual_controls', 'raw_format_support', 'burst_speed_fps', 'weather_sealed', 'video_resolution_4k']


Model Configuration

In [ ]:
# Model + sampling configuration
# (MODEL_NAME is defined in cell 3 — do not redefine here)

MODEL_OPTIONS = {
    "temperature": 0.75,      # >0 intentional: enables variation across iterations
    "top_p": 0.95,
    "max_output_tokens": 1024, # raised from 512 to avoid truncation on reasoning models
}

N_ITERATIONS = 10   # 10 iterations per persona for statistical power

# ============================================================
#               TEAM PARALLELIZATION CONFIG
# ============================================================
# Each teammate runs this notebook in their own Colab session
# with a UNIQUE RUNNER_ID. The 14 personas are split into
# NUM_RUNNERS contiguous chunks; each runner executes one chunk.
#
# After all runners finish, gather every per-runner output file
# (llm_persona_outputs_runner1.json, runner2.json, etc.)
# into one folder and run the MERGE cell at the bottom to
# produce the combined results file.
#
# >>>>>>>>>  CHANGE THE TWO LINES BELOW PER TEAMMATE  <<<<<<<<<
NUM_RUNNERS = 5   # Total teammates running the notebook
RUNNER_ID   = 1   # YOUR runner number — must be unique 1..NUM_RUNNERS
# ============================================================

print("Model:", MODEL_NAME)
print("Iterations per persona:", N_ITERATIONS)
print("Options:", MODEL_OPTIONS)
print(f"Runner: {RUNNER_ID} of {NUM_RUNNERS}")

In [ ]:
import json
import re
import time
import math

# ----------------------------
# Load camera pair data
# ----------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


SIMPLE_DATA = load_json(DATA_DIR / "cameras_simple.json")
MODERATE_DATA = load_json(DATA_DIR / "cameras_moderate.json")
COMPLEX_DATA = load_json(DATA_DIR / "cameras_complex.json")

SIMPLE_PAIRS = SIMPLE_DATA["pairs"]
MODERATE_PAIRS = MODERATE_DATA["pairs"]
COMPLEX_PAIRS = COMPLEX_DATA["pairs"]

print(f"Personas:       {len(ALL_PERSONAS)}")
print(f"Simple pairs:   {len(SIMPLE_PAIRS)}")
print(f"Moderate pairs: {len(MODERATE_PAIRS)}")
print(f"Complex pairs:  {len(COMPLEX_PAIRS)}")

# ----------------------------
# Persona slice for THIS runner
# ----------------------------
# 14 personas split into NUM_RUNNERS contiguous chunks (last chunk may be smaller).
if not (1 <= RUNNER_ID <= NUM_RUNNERS):
    raise ValueError(f"RUNNER_ID must be between 1 and {NUM_RUNNERS}, got {RUNNER_ID}")

_chunk_size = math.ceil(len(ALL_PERSONAS) / NUM_RUNNERS)
_start_idx  = (RUNNER_ID - 1) * _chunk_size
_end_idx    = _start_idx + _chunk_size
MY_PERSONAS = ALL_PERSONAS[_start_idx:_end_idx]

print(f"\nRunner {RUNNER_ID}/{NUM_RUNNERS} - assigned {len(MY_PERSONAS)} personas:")
for p in MY_PERSONAS:
    print(f"  {p['id']:6}  {p['label']:6}  {p['name']}")
print(f"Will produce {len(MY_PERSONAS) * N_ITERATIONS} sessions total\n")


# ----------------------------
# Prompt builders
# ----------------------------
def build_ranking_prompt():
    lines = [f"  - {attr}: {ATTRIBUTE_DESCRIPTIONS[attr]}" for attr in ATTRIBUTE_POOL]
    attrs_block = "\n".join(lines)
    return (
        "You are considering purchasing a new digital camera. Before seeing any "
        "specific products, please rank the following camera attributes from most "
        "to least important based on YOUR personal priorities.\n\n"
        f"The attributes are:\n{attrs_block}\n\n"
        "Return your ranking as a numbered list, with one attribute name per line, "
        "from most important (1) to least important. Use the exact attribute names "
        "as shown above (the part before the colon).\n\n"
        "Example format:\n"
        "1. attribute_name\n"
        "2. attribute_name\n"
        "3. attribute_name\n\n"
        "Your ranked list:"
    )


def format_camera(camera, pair_attributes):
    lines = [f"  {camera['name']}"]
    for attr in pair_attributes:
        desc = ATTRIBUTE_DESCRIPTIONS[attr]
        lines.append(f"    - {desc}: {camera[attr]}")
    return "\n".join(lines)


def build_pair_prompt(pair):
    pair_attrs = pair["pair_attributes"]
    a_text = format_camera(pair["camera_a"], pair_attrs)
    b_text = format_camera(pair["camera_b"], pair_attrs)
    return (
        "Based on your priority ranking above, evaluate this next pair of cameras.\n\n"
        f"Camera A:\n{a_text}\n\n"
        f"Camera B:\n{b_text}\n\n"
        "Which camera do you choose? Reply in exactly this format:\n\n"
        "Choice: [camera name]\n\n"
        "Rule: You must make a choice. Use the exact camera name."
    )


# ----------------------------
# Gemini multi-turn chat call (with retry + exponential backoff)
# ----------------------------
# Gemini uses a different message format than OpenAI:
#   - System prompt is passed as system_instruction to the model constructor
#   - Roles are "user" and "model" (not "assistant")
#   - Each message is {"role": ..., "parts": [{"text": ...}]}
#
# We rebuild the Gemini model object per session so the system_instruction
# (persona) is correctly set for each persona's conversation.

def build_gemini_history(messages):
    """Convert OpenAI-style messages to Gemini chat history format.

    The system message is handled separately via system_instruction,
    so it is excluded from the history here.

    Args:
        messages: list of {role, content} dicts

    Returns:
        tuple: (system_instruction_text, gemini_history_list)
    """
    system_instruction = ""
    history = []
    for msg in messages:
        if msg["role"] == "system":
            system_instruction = msg["content"]
        elif msg["role"] == "user":
            history.append({"role": "user", "parts": [{"text": msg["content"]}]})
        elif msg["role"] == "assistant":
            # Gemini uses "model" instead of "assistant"
            history.append({"role": "model", "parts": [{"text": msg["content"]}]})
    return system_instruction, history


def chat_call(messages, retries=5):
    """Call the Gemini API with automatic retry on transient failures.

    Respects the suggested retry delay from 429 rate-limit responses
    instead of using fixed exponential backoff, so the code does not
    hammer the API and waste retries.

    Args:
        messages: list of {role, content} dicts (full conversation history)
        retries:  number of attempts before raising (default 5)

    Returns:
        str: model reply text
    """
    system_instruction, history = build_gemini_history(messages)

    if not history or history[-1]["role"] != "user":
        raise ValueError("Last message must be from the user.")

    chat_history = history[:-1]                        # all turns before the latest
    user_input   = history[-1]["parts"][0]["text"]     # the latest user turn

    generation_config = genai.types.GenerationConfig(
        temperature=MODEL_OPTIONS["temperature"],
        top_p=MODEL_OPTIONS["top_p"],
        max_output_tokens=MODEL_OPTIONS["max_output_tokens"],
    )

    # Create the model object ONCE outside the retry loop — no need to
    # reconstruct it on every attempt since model config doesn't change.
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=system_instruction,
        generation_config=generation_config,
    )

    for attempt in range(retries):
        try:
            chat = model.start_chat(history=chat_history)
            response = chat.send_message(user_input)
            return response.text.strip()

        except Exception as e:
            error_str = str(e)

            # Never retry on auth / bad-request errors — they won't resolve themselves
            if any(code in error_str for code in ["400", "401", "403"]):
                print(f"  Non-retryable error: {e}")
                raise

            if attempt < retries - 1:
                # For 429 rate-limit errors, honour the API's suggested wait time
                if "429" in error_str:
                    match = re.search(r"retry in ([\d\.]+)s", error_str, re.IGNORECASE)
                    wait = float(match.group(1)) if match else 60
                    print(f"  Rate limited (attempt {attempt + 1}/{retries}). "
                          f"Waiting {wait:.0f}s as suggested by API...")
                else:
                    wait = 2 ** attempt   # 1s, 2s, 4s, 8s for other transient errors
                    print(f"  API error (attempt {attempt + 1}/{retries}): {e}. "
                          f"Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  API call failed after {retries} attempts: {e}")
                raise

In [ ]:
# ----------------------------
# Response parsers
# ----------------------------
RANK_LINE_RE = re.compile(r"^\s*\d+\s*[\.\):\-]\s*[`*_]*([a-zA-Z][a-zA-Z0-9_ ]*)")


def parse_ranking(text, attribute_pool):
    """Parse a numbered list into an ordered list of attribute keys."""
    pool_set = set(attribute_pool)
    order = []
    seen = set()
    for line in text.splitlines():
        m = RANK_LINE_RE.match(line)
        if not m:
            continue
        token = m.group(1).strip().strip("`*_").lower().replace(" ", "_")
        # Handle trailing descriptions after the attribute key.
        # Also match when the token is a prefix of the attribute (e.g.
        # "ease_of_use" should match "ease_of_use_score").
        for attr in pool_set:
            if token == attr or token.startswith(attr + "_") or token.startswith(attr) or attr.startswith(token):
                if attr not in seen:
                    order.append(attr)
                    seen.add(attr)
                break
    # Append any attributes the model missed so the ranking stays total.
    for attr in attribute_pool:
        if attr not in seen:
            order.append(attr)
    return order


def parse_choice(text, camera_a_name, camera_b_name, attribute_pool):
    choice_letter = ""

    choice_match = re.search(r"choice\s*:\s*(.+)", text, re.IGNORECASE)
    if choice_match:
        raw = choice_match.group(1).splitlines()[0].strip().strip("*`[]()")
        raw_lower = raw.lower()
        if camera_a_name.lower() in raw_lower:
            choice_letter = "A"
        elif camera_b_name.lower() in raw_lower:
            choice_letter = "B"
        # Handle "Camera A" / "Camera B" literal responses
        elif re.search(r"\bcamera\s+a\b", raw_lower):
            choice_letter = "A"
        elif re.search(r"\bcamera\s+b\b", raw_lower):
            choice_letter = "B"
        elif re.match(r"^\s*a\b", raw_lower):
            choice_letter = "A"
        elif re.match(r"^\s*b\b", raw_lower):
            choice_letter = "B"

    return choice_letter

In [ ]:
# ----------------------------
# Session runner (single multi-turn conversation)
# ----------------------------
def run_session(persona, run_id):
    result = {
        "persona_id": persona["id"],
        "persona_name": persona["name"],
        "persona_label": persona["label"],
        "generation_id": run_id,
        "priority_order": [],
        "simple": {},
        "moderate": {},
        "complex": {},
    }

    messages = [{"role": "system", "content": persona["system_prompt"]}]

    # Turn 1: priority ranking over all 11 attributes
    ranking_prompt = build_ranking_prompt()
    messages.append({"role": "user", "content": ranking_prompt})
    ranking_text = chat_call(messages, retries=5)
    messages.append({"role": "assistant", "content": ranking_text})
    result["priority_order"] = parse_ranking(ranking_text, ATTRIBUTE_POOL)
    result["priority_order_raw"] = ranking_text

    # Turns 2-16: 5 simple -> 5 moderate -> 5 complex
    levels = [
        ("simple", SIMPLE_PAIRS),
        ("moderate", MODERATE_PAIRS),
        ("complex", COMPLEX_PAIRS),
    ]
    for level_name, pairs in levels:
        for pair in pairs:
            pair_prompt = build_pair_prompt(pair)
            messages.append({"role": "user", "content": pair_prompt})
            pair_response = chat_call(messages, retries=5)
            messages.append({"role": "assistant", "content": pair_response})
            choice = parse_choice(
                pair_response,
                pair["camera_a"]["name"],
                pair["camera_b"]["name"],
                ATTRIBUTE_POOL,
            )
            result[level_name][str(pair["pair_id"])] = {
                "choice": choice,
                "raw_response": pair_response,
            }
            time.sleep(0.3)  # brief pause between API calls to avoid rate limits

    return result

In [ ]:
# ----------------------------
# Main loop (with resume support)
# ----------------------------
# If the output file already exists (e.g. from a previous run or a crash),
# completed sessions are loaded and skipped so you don't re-run or overwrite
# them. This is critical for long runs (N_ITERATIONS=30) where Colab
# disconnects are common.

def main():
    output_path = OUTPUT_DIR / f"llm_persona_outputs_runner{RUNNER_ID}.json"

    # --- Resume: load existing results if the file exists ---
    if output_path.exists():
        try:
            with open(output_path, "r", encoding="utf-8") as f:
                all_results = json.load(f)
            completed = {(r["persona_id"], r["generation_id"]) for r in all_results}
            print(f"Resuming: loaded {len(all_results)} existing sessions")
        except (json.JSONDecodeError, KeyError):
            print("WARNING: existing output file is corrupted, starting fresh")
            all_results = []
            completed = set()
    else:
        all_results = []
        completed = set()

    total = len(MY_PERSONAS) * N_ITERATIONS
    remaining = total - len([1 for p in MY_PERSONAS
                             for r in range(1, N_ITERATIONS + 1)
                             if (p["id"], r) in completed])
    print(f"Total sessions: {total} | Already done: {total - remaining} | Remaining: {remaining}")

    for persona in MY_PERSONAS:
        for run_id in range(1, N_ITERATIONS + 1):
            if (persona["id"], run_id) in completed:
                continue

            print(f"Running {persona['id']} ({persona['label']}) | run {run_id}")

            # Session-level retry: 2 attempts before giving up on this session
            success = False
            for session_attempt in range(2):
                try:
                    session_result = run_session(persona, run_id)
                    all_results.append(session_result)
                    completed.add((persona["id"], run_id))

                    # Checkpoint: save after every successful session
                    with open(output_path, "w", encoding="utf-8") as f:
                        json.dump(all_results, f, indent=2, ensure_ascii=False)

                    success = True
                    break
                except Exception as e:
                    if session_attempt == 0:
                        print(f"  Error (attempt 1/2): {e}")
                        print(f"  Retrying in 10s...")
                        time.sleep(10)
                    else:
                        print(f"  Error (attempt 2/2): {e} -- skipping this session")

            if not success:
                print(f"  SKIPPED {persona['id']} run {run_id}")

    print(f"\nDone. Saved {len(all_results)} sessions to: {output_path}")


if __name__ == "__main__":
    main()

In [ ]:
# ----------------------------
# Auto-push results to GitHub
# ----------------------------
# Uses the token from cell 1 (already embedded in the clone URL).
import subprocess

os_result = subprocess.run(
    ["git", "add", f"results/llm_persona_outputs_runner{RUNNER_ID}.json"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

os_result = subprocess.run(
    ["git", "-c", "user.name=Colab Runner", "-c", "user.email=noreply@colab",
     "commit", "-m", f"Add runner {RUNNER_ID} results ({N_ITERATIONS} iterations)"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

os_result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

if os_result.returncode == 0:
    print(f"Pushed runner {RUNNER_ID} results to GitHub.")
else:
    print("Push failed — you may need to pull first or push manually.")

In [ ]:
import json
with open("/content/cs568-project/results/llm_persona_outputs_runner1.json") as f:
    print(json.load(f))

[{'persona_id': 'EXP_01', 'persona_name': 'Marcus Chen', 'persona_label': 'expert', 'generation_id': 1, 'priority_order': ['raw_format_support', 'image_quality_score', 'has_manual_controls', 'weather_sealed', 'price_usd', 'battery_life_hours', 'ease_of_use_score', 'burst_speed_fps', 'video_resolution_4k', 'weight_grams', 'optical_zoom_x'], 'simple': {'1': {'choice': 'B', 'raw_response': 'Choice: Brikon S2'}, '2': {'choice': 'B', 'raw_response': 'Choice: Dunox P4'}, '3': {'choice': 'B', 'raw_response': 'Choice: Fluma K6'}, '4': {'choice': 'B', 'raw_response': 'Choice: Helion Q8'}, '5': {'choice': 'B', 'raw_response': 'Choice: Joltex C3'}}, 'moderate': {'1': {'choice': 'B', 'raw_response': 'Choice: Lumion B2'}, '2': {'choice': 'B', 'raw_response': 'Choice: Noxar D4'}, '3': {'choice': 'B', 'raw_response': 'Choice: Pryvex F6'}, '4': {'choice': 'A', 'raw_response': 'Choice: Quasar G7'}, '5': {'choice': 'B', 'raw_response': 'Choice: Trevon J1'}}, 'complex': {'1': {'choice': 'B', 'raw_respons

In [ ]:
# ----------------------------
# Pull latest results from all runners
# ----------------------------
# Run this BEFORE the merge cell so you have everyone's runner files locally.
import subprocess

os_result = subprocess.run(
    ["git", "pull", "origin", "main"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

if os_result.returncode == 0:
    print("Pulled latest from GitHub.")
else:
    print("Pull failed — check for conflicts or auth issues.")

In [ ]:
# ============================================================
#  MERGE HELPER - run AFTER all teammates have finished
# ============================================================
# Place every per-runner output file into MERGE_INPUT_DIR before
# running this cell. By default this looks in OUTPUT_DIR
# (/content/cs568-project/results), but you can change the path
# below to a Google Drive folder, a git checkout, etc.
#
# This cell is only run by ONE teammate (the one assembling the
# combined dataset for downstream scoring). The other teammates
# do NOT need to run this cell.

import json
import glob
from pathlib import Path

MERGE_INPUT_DIR = OUTPUT_DIR   # change if files were collected elsewhere
MERGE_OUTPUT    = OUTPUT_DIR / "llm_persona_outputs_merged.json"

runner_files = sorted(glob.glob(str(MERGE_INPUT_DIR / "llm_persona_outputs_runner*.json")))
print(f"Found {len(runner_files)} runner files:")
for f in runner_files:
    print(f"  {f}")

merged = []
seen = set()
duplicates = []
for f in runner_files:
    with open(f, "r", encoding="utf-8") as fp:
        sessions = json.load(fp)
    for s in sessions:
        key = (s["persona_id"], s["generation_id"])
        if key in seen:
            duplicates.append(key)
            continue
        seen.add(key)
        merged.append(s)

# Sanity check: which personas are present? which are missing?
expected_personas = {p["id"] for p in ALL_PERSONAS}
got_personas = {s["persona_id"] for s in merged}
missing = expected_personas - got_personas

print(f"\nMerged {len(merged)} sessions from {len(got_personas)}/{len(expected_personas)} personas")
if missing:
    print(f"WARNING: Missing personas: {sorted(missing)}")
if duplicates:
    print(f"WARNING: Skipped {len(duplicates)} duplicate (persona, gen) pairs (first 5): {duplicates[:5]}")

with open(MERGE_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)
print(f"\nWrote combined file: {MERGE_OUTPUT}")


In [ ]:
# ----------------------------
# Push merged results to GitHub
# ----------------------------
import subprocess

os_result = subprocess.run(
    ["git", "add", "results/llm_persona_outputs_merged.json"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

os_result = subprocess.run(
    ["git", "-c", "user.name=Colab Runner", "-c", "user.email=noreply@colab",
     "commit", "-m", "Add merged results (all runners combined)"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

os_result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True
)
print(os_result.stdout + os_result.stderr)

if os_result.returncode == 0:
    print("Pushed merged results to GitHub.")
else:
    print("Push failed — you may need to pull first or push manually.")